---
## Cell 0 — Поиск гиперпараметров (11 экспериментов)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import pandas as pd
import time
import os
from tqdm import tqdm
from pathlib import Path

# ==========================================
# 🔧 КОНФИГУРАЦИЯ
# ==========================================
# [ЗАМЕЧАНИЕ]: DATA_ROOT и SAVE_DIR задаются в каждой из 4 ячеек отдельно.
# Вынеси в отдельную конфигурационную ячейку в самом начале ноутбука.
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./night_results")
SAVE_DIR.mkdir(exist_ok=True)

MODALITIES = ["ДС", "УФ"]
EPOCHS = 10  # Можно снизить до 5 для ускорения ночного теста
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 🔹 МАТРИЦА ИЗ 11 МОДЕЛЕЙ (1 референс + 10 вариаций)
EXPERIMENTS = [
    {"id": "0_Reference",      "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "1_LR_Low",         "lr": 5e-5, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "2_LR_High",        "lr": 5e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "3_WD_Low",         "lr": 1e-4, "wd": 1e-5, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "4_WD_High",        "lr": 1e-4, "wd": 1e-3, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "5_Drop_Low",       "lr": 1e-4, "wd": 1e-4, "dropout": 0.3, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "6_Drop_High",      "lr": 1e-4, "wd": 1e-4, "dropout": 0.7, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "7_Opt_SGD",        "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "sgd",   "sched": "plateau", "resize": "pad",   "aug": "std"},
    {"id": "8_Sched_Cosine",   "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "cosine",  "resize": "pad",   "aug": "std"},
    {"id": "9_Aug_Heavy",      "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",   "aug": "heavy"},
    {"id": "10_Resize_Crop",   "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "crop",  "aug": "std"},
]

# ==========================================
# 🖼️ ТРАНСФОРМЫ
# ==========================================
class PadToSquare:
    def __init__(self, fill=128): self.fill = fill
    def __call__(self, img):
        w, h = img.size
        if w == h: return img
        pad = abs(w - h) // 2
        return transforms.Pad((0, pad, 0, pad) if w > h else (pad, 0, pad, 0), fill=self.fill)(img)

# [ОШИБКА]: Неправильный порядок аугментаций — эта проблема есть во всех 4 ячейках.
# Аугментации применяются ДО resize.
# Правильный порядок: сначала PadToSquare + Resize до 224×224, затем аугментация.
def get_transforms(resize_strategy, aug_level, is_train):
    ops = []
    if is_train:
        if aug_level == "heavy":
            ops.append(transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2))
            ops.append(transforms.RandomRotation(15))
        else:  # std
            ops.append(transforms.ColorJitter(brightness=0.2, contrast=0.2))
        ops.append(transforms.RandomHorizontalFlip(p=0.5))
        ops.append(transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)))

    if resize_strategy == 'pad':
        ops.append(PadToSquare(fill=128))
        ops.append(transforms.Resize((224, 224)))
    elif resize_strategy == 'crop':
        if is_train:
            ops.append(transforms.RandomResizedCrop(224, scale=(0.85, 1.0)))
        else:
            ops.append(transforms.Resize(256))
            ops.append(transforms.CenterCrop(224))

    ops.append(transforms.ToTensor())
    ops.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    return transforms.Compose(ops)

# ==========================================
# 📦 DATALOADER
# ==========================================
def prepare_loaders(modality, resize_strategy, aug_level):
    mod_path = DATA_ROOT / modality
    train_ds = datasets.ImageFolder(mod_path / "train", transform=get_transforms(resize_strategy, aug_level, True))
    val_ds   = datasets.ImageFolder(mod_path / "val",   transform=get_transforms(resize_strategy, aug_level, False))

    class_counts = torch.bincount(torch.tensor(train_ds.targets))
    sample_weights = [1.0 / class_counts[label] for label in train_ds.targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
    return train_loader, val_loader, train_ds.classes

# ==========================================
# 🧠 МОДЕЛЬ
# ==========================================
def create_model(num_classes, dropout_p):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(
        nn.Dropout(dropout_p),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model.to(DEVICE)

# ==========================================
# 🔄 ОБУЧЕНИЕ И ВАЛИДАЦИЯ (5 МЕТРИК)
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, epoch, total_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc=f"Train {epoch}/{total_epochs}", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return running_loss / len(loader.dataset)

def validate(model, loader, criterion, epoch, total_epochs):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc=f"Val   {epoch}/{total_epochs}", leave=False)
    with torch.no_grad():
        for inputs, labels in pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    return {
        "loss": val_loss,
        "f1": f1_score(all_labels, all_preds, average='macro', zero_division=0),
        "acc": accuracy_score(all_labels, all_preds),
        "prec": precision_score(all_labels, all_preds, average='macro', zero_division=0),
        "rec": recall_score(all_labels, all_preds, average='macro', zero_division=0)
    }

# ==========================================
# 🧪 ЗАПУСК ЭКСПЕРИМЕНТА
# ==========================================
def run_experiment(modality, cfg):
    # [TODO]: добавить set_seed(42) в начало — без фиксации seed порядок батчей
    # в WeightedRandomSampler будет разным при каждом запуске.
    # В ячейках 2 и 3 это уже сделано.
    print(f"\n🚀 {cfg['id']} | {modality} | LR:{cfg['lr']:.0e} WD:{cfg['wd']:.0e} Drop:{cfg['dropout']} Opt:{cfg['opt']} Sched:{cfg['sched']}")
    print("-" * 80)

    train_loader, val_loader, classes = prepare_loaders(modality, cfg['resize'], cfg['aug'])
    model = create_model(len(classes), cfg['dropout'])
    criterion = nn.CrossEntropyLoss()  # Без весов, т.к. есть WeightedRandomSampler

    if cfg['opt'] == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    elif cfg['opt'] == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['wd'])

    if cfg['sched'] == 'plateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    elif cfg['sched'] == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_metrics = {"f1": 0.0, "epoch": 0}
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, epoch, EPOCHS)
        val_mets = validate(model, val_loader, criterion, epoch, EPOCHS)
        # [ЗАМЕЧАНИЕ]: scheduler.step(val_mets['f1']) правильно для ReduceLROnPlateau.
        # Для CosineAnnealingLR аргумент не нужен — вызывай scheduler.step() без аргументов.
        # Добавь ветку: if cfg['sched'] == 'plateau': scheduler.step(val_mets['f1'])
        #               else: scheduler.step()
        scheduler.step(val_mets['f1'])

        if val_mets['f1'] > best_metrics['f1']:
            best_metrics = {**val_mets, "epoch": epoch}
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_{modality}_best.pth")

        print(f"✅ Ep {epoch:2d} | TrL:{train_loss:.4f} | VL:{val_mets['loss']:.4f} F1:{val_mets['f1']:.4f} Acc:{val_mets['acc']:.4f} | {time.time()-t0:.1f}s")
        history.append({"epoch": epoch, **val_mets})
    # [ЗАМЕЧАНИЕ]: history накапливается, но не сохраняется и не возвращается из функции.
    # После обучения построить кривые обучения невозможно.
    # Добавь сохранение: json.dump(history, open(SAVE_DIR / f"{cfg['id']}_{modality}_history.json", "w"))

    torch.cuda.empty_cache()
    return {"config": cfg, "modality": modality, "best": best_metrics}

# ==========================================
# 📊 СВОДНАЯ ТАБЛИЦА И АНАЛИЗ
# ==========================================
if __name__ == "__main__":
    print(f"🖥️ Device: {DEVICE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")
    print(f"📂 Data: {DATA_ROOT}\n")

    all_results = []
    for mod in MODALITIES:
        for cfg in EXPERIMENTS:
            res = run_experiment(mod, cfg)
            all_results.append(res)

    # Формируем DataFrame
    rows = []
    for r in all_results:
        c, m, b = r['config'], r['modality'], r['best']
        rows.append({
            "Model": c['id'], "Modality": m,
            "LR": f"{c['lr']:.0e}", "WD": f"{c['wd']:.0e}", "Dropout": c['dropout'],
            # [ЗАМЕЧАНИЕ]: ключи 'Optimizer' и 'Scheduler' — не 'Opt' и 'Sched'.
            # В display_cols ниже используются неверные имена.
            "Optimizer": c['opt'], "Scheduler": c['sched'], "Resize": c['resize'], "Aug": c['aug'],
            "Best_Epoch": b['epoch'],
            "Val_F1": b['f1'], "Val_Acc": b['acc'], "Val_Prec": b['prec'], "Val_Rec": b['rec'], "Val_Loss": b['loss']
        })
    df = pd.DataFrame(rows)

    # Считаем дельты относительно референса (0_Reference)
    for mod in MODALITIES:
        ref_row = df[(df['Model'] == '0_Reference') & (df['Modality'] == mod)].iloc[0]
        mask = df['Modality'] == mod
        df.loc[mask, 'ΔF1'] = (df.loc[mask, 'Val_F1'] - ref_row['Val_F1']).round(4)
        # [ОШИБКА]: KeyError — колонок 'Acc', 'Prec', 'Rec', 'Loss' нет в DataFrame.
        # В rows выше они называются 'Val_Acc', 'Val_Prec', 'Val_Rec', 'Val_Loss'.
        # Исправить: ref_row['Val_Acc'], ref_row['Val_Prec'], ref_row['Val_Rec'], ref_row['Val_Loss']
        df.loc[mask, 'ΔAcc'] = (df.loc[mask, 'Val_Acc'] - ref_row['Acc']).round(4)
        df.loc[mask, 'ΔPrec'] = (df.loc[mask, 'Val_Prec'] - ref_row['Prec']).round(4)
        df.loc[mask, 'ΔRec'] = (df.loc[mask, 'Val_Rec'] - ref_row['Rec']).round(4)
        df.loc[mask, 'ΔLoss'] = (df.loc[mask, 'Val_Loss'] - ref_row['Loss']).round(4)

    # Сортируем и выводим
    df = df.sort_values(['Modality', 'Model'])
    # [ОШИБКА]: 'Opt' и 'Sched' — таких колонок нет в DataFrame.
    # Колонки называются 'Optimizer' и 'Scheduler' (см. rows выше).
    # Исправить: заменить 'Opt' → 'Optimizer', 'Sched' → 'Scheduler'
    display_cols = ['Model', 'Modality', 'LR', 'WD', 'Dropout', 'Opt', 'Sched', 'Resize', 'Aug',
                    'Val_F1', 'ΔF1', 'Val_Acc', 'ΔAcc', 'Val_Prec', 'ΔPrec', 'Val_Rec', 'ΔRec', 'Val_Loss', 'ΔLoss']

    print("\n" + "="*120)
    print("📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА (Δ = отклонение от 0_Reference)")
    print("="*120)
    pd.options.display.max_columns = None
    pd.options.display.width = 200
    print(df[display_cols].to_string(index=False))

    df.to_csv(SAVE_DIR / "full_comparison.csv", index=False)
    print(f"\n💾 Полная таблица сохранена: {SAVE_DIR / 'full_comparison.csv'}")
    print("🌙 Тест завершен. Анализируй дельты: положительные Δ означают улучшение относительно референса.")

---
## Cell 1 — Эксперименты с заморозкой слоёв

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd
import time
from tqdm import tqdm  # 🔹 Простой импорт — работает везде
from pathlib import Path

# ==========================================
# 🔧 КОНФИГУРАЦИЯ
# ==========================================
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./freeze_test_best")
SAVE_DIR.mkdir(exist_ok=True)

PATIENCE = 5
MIN_DELTA = 0.005
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 📋 ЭКСПЕРИМЕНТЫ
# ==========================================
EXPERIMENTS = [
    {"id": "A_DS_None",    "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3},
    {"id": "B_DS_Partial", "mod": "ДС", "freeze": "partial", "lr": 5e-5, "wd": 1e-3},
    {"id": "C_DS_Full",    "mod": "ДС", "freeze": "full",    "lr": 1e-3, "wd": 1e-3},
    {"id": "D_UF_None",    "mod": "УФ", "freeze": "none",    "lr": 1e-4, "wd": 1e-5},
    {"id": "E_UF_Partial", "mod": "УФ", "freeze": "partial", "lr": 1e-4, "wd": 1e-5},
    {"id": "F_UF_Full",    "mod": "УФ", "freeze": "full",    "lr": 1e-3, "wd": 1e-5},
]

# ==========================================
# 🖼️ ТРАНСФОРМЫ
# ==========================================
class PadToSquare:
    def __init__(self, fill=128): self.fill = fill
    def __call__(self, img):
        w, h = img.size
        if w == h: return img
        pad = abs(w - h) // 2
        return transforms.Pad((0, pad, 0, pad) if w > h else (pad, 0, pad, 0), fill=self.fill)(img)

def get_transforms(is_train):
    ops = []
    if is_train:
        ops.append(transforms.ColorJitter(brightness=0.2, contrast=0.2))
        ops.append(transforms.RandomHorizontalFlip(p=0.5))
        ops.append(transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)))
    ops.append(PadToSquare(fill=128))
    ops.append(transforms.Resize((224, 224)))
    ops.append(transforms.ToTensor())
    ops.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    return transforms.Compose(ops)

# ==========================================
# 🧠 МОДЕЛЬ
# ==========================================
def create_model(num_classes, freeze_mode):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_mode == "partial":
        for name, param in model.named_parameters():
            if any(x in name for x in ["conv1", "bn1", "layer1", "layer2"]):
                param.requires_grad = False
    elif freeze_mode == "full":
        for name, param in model.named_parameters():
            if "layer4" not in name and "fc" not in name:
                param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

# ==========================================
# 🔄 ОБУЧЕНИЕ С ПРОГРЕСС-БАРАМИ
# ==========================================
def run_experiment(cfg):
    print(f"\n🚀 {cfg['id']} | Freeze: {cfg['freeze']} | {cfg['mod']}")

    mod_path = DATA_ROOT / cfg['mod']
    train_ds = datasets.ImageFolder(mod_path / "train", transform=get_transforms(True))
    val_ds   = datasets.ImageFolder(mod_path / "val",   transform=get_transforms(False))

    class_counts = torch.bincount(torch.tensor(train_ds.targets))
    sample_weights = [1.0 / class_counts[t] for t in train_ds.targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    model = create_model(len(train_ds.classes), cfg['freeze'])
    optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['wd'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    criterion = nn.CrossEntropyLoss()

    best_f1, best_epoch, patience_cnt = 0.0, 0, 0

    for epoch in range(1, 51):
        # === 🔹 TRAIN с прогресс-баром ===
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Ep {epoch} Train", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        train_loss = running_loss / len(train_ds)

        # === 🔹 VALIDATE с прогресс-баром ===
        model.eval()
        val_loss, preds, labels_list = 0.0, [], []
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Ep {epoch} Val", leave=False):
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
                _, p = torch.max(outputs, 1)
                preds.extend(p.cpu().numpy())
                labels_list.extend(labels.cpu().numpy())
        val_loss /= len(val_ds)
        f1 = f1_score(labels_list, preds, average='macro', zero_division=0)
        scheduler.step()

        # === 🔹 EARLY STOPPING ===
        if f1 > best_f1 + MIN_DELTA:
            best_f1, best_epoch, patience_cnt = f1, epoch, 0
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_best.pth")
            status = "⭐ BEST"
        else:
            patience_cnt += 1
            status = f"wait {patience_cnt}/{PATIENCE}"

        print(f"✅ Ep {epoch:2d} | TrL: {train_loss:.4f} | VL: {val_loss:.4f} | F1: {f1:.4f} {status}")

        if patience_cnt >= PATIENCE:
            print(f"⏹️ Early Stop at epoch {epoch}. Best F1: {best_f1:.4f}")
            break

    torch.cuda.empty_cache()
    return {"id": cfg['id'], "modality": cfg['mod'], "freeze": cfg['freeze'],
            "best_f1": round(best_f1, 4), "best_epoch": best_epoch}

# ==========================================
# 🏁 ЗАПУСК
# ==========================================
if __name__ == "__main__":
    print(f"🖥️ Device: {DEVICE}")
    results = []
    for cfg in EXPERIMENTS:
        res = run_experiment(cfg)
        results.append(res)

    df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("📊 ИТОГИ")
    print("="*80)
    for mod in ["ДС", "УФ"]:
        print(f"\n🔹 Модальность: {mod}")
        mod_df = df[df['modality'] == mod].sort_values('best_f1', ascending=False)
        print(mod_df[['id', 'freeze', 'best_f1', 'best_epoch']].to_string(index=False))

    df.to_csv(SAVE_DIR / "freeze_comparison.csv", index=False)
    print(f"\n💾 Таблица: {SAVE_DIR / 'freeze_comparison.csv'}")

---
## Cell 2 — Аблационное исследование (FocalLoss / LabelSmooth / CutMix)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
import random
import time
from tqdm import tqdm
from pathlib import Path

# ==========================================
# 🔒 1. ФИКСАЦИЯ СИДА
# ==========================================
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return torch.Generator().manual_seed(seed)

print(f"🔒 SEED={SEED} | Все источники случайности зафиксированы")

# ==========================================
# 🔧 2. КОНФИГУРАЦИЯ (СТРОГОЕ A/B ТЕСТИРОВАНИЕ)
# ==========================================
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./ablation_strict")
SAVE_DIR.mkdir(exist_ok=True)

PATIENCE = 5
MIN_DELTA = 0.005
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_EPOCHS = 50

# 📋 Матрица: БАЗА + 1 изменение на модель
EXPERIMENTS = [
    # === ДС: БАЗА (LR=5e-5, WD=1e-3, Drop=0.3, SGD, Cosine, Heavy, Pad, none) ===
    {"id": "DS_Base",          "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3,
     "loss": "ce", "mix": "none", "aug": "heavy", "resize": "pad"},

    {"id": "DS_FocalLoss",     "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3,
     "loss": "focal", "mix": "none", "aug": "heavy", "resize": "pad"},  # 🆕 Только loss

    {"id": "DS_LabelSmooth",   "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3,
     "loss": "label_smooth", "mix": "none", "aug": "heavy", "resize": "pad"},  # 🆕 Только loss

    {"id": "DS_CutMix",        "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3,
     "loss": "ce", "mix": "cutmix", "aug": "heavy", "resize": "pad"},  # 🆕 Только mix

    # === УФ: БАЗА (LR=5e-4, WD=1e-5, Drop=0.3, SGD, Cosine, Std, Crop, partial) ===
    {"id": "UF_Base",          "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3,
     "loss": "ce", "mix": "none", "aug": "std", "resize": "crop"},

    {"id": "UF_FocalLoss",     "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3,
     "loss": "focal", "mix": "none", "aug": "std", "resize": "crop"},  # 🆕 Только loss

    {"id": "UF_LabelSmooth",   "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3,
     "loss": "label_smooth", "mix": "none", "aug": "std", "resize": "crop"},  # 🆕 Только loss

    {"id": "UF_CutMix",        "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3,
     "loss": "ce", "mix": "cutmix", "aug": "std", "resize": "crop"},  # 🆕 Только mix
]

# ==========================================
# 🎨 3. ТРАНСФОРМЫ
# ==========================================
class PadToSquare:
    def __init__(self, fill=128): self.fill = fill
    def __call__(self, img):
        w, h = img.size
        if w == h: return img
        pad = abs(w - h) // 2
        return transforms.Pad((0, pad, 0, pad) if w > h else (pad, 0, pad, 0), fill=self.fill)(img)

def get_transforms(cfg, is_train):
    ops = []
    if is_train:
        if cfg['aug'] == 'heavy':
            ops.append(transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2))
            ops.append(transforms.RandomRotation(15))
        else:  # std
            ops.append(transforms.ColorJitter(brightness=0.2, contrast=0.2))
        ops.append(transforms.RandomHorizontalFlip(p=0.5))
        ops.append(transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)))

    if cfg['resize'] == 'pad':
        ops.append(PadToSquare(fill=128))
        ops.append(transforms.Resize((224, 224)))
    elif cfg['resize'] == 'crop':
        if is_train:
            ops.append(transforms.RandomResizedCrop(224, scale=(0.85, 1.0)))
        else:
            ops.append(transforms.Resize(256))
            ops.append(transforms.CenterCrop(224))

    ops.append(transforms.ToTensor())
    ops.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    return transforms.Compose(ops)

# ==========================================
# 🧠 4. МОДЕЛЬ + ЗАМОРОЗКА
# ==========================================
def create_model(num_classes, freeze_mode, dropout_p):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_mode == "partial":
        for n, p in model.named_parameters():
            if any(x in n for x in ["conv1", "bn1", "layer1", "layer2"]):
                p.requires_grad = False
    # freeze="none" — всё обучается

    model.fc = nn.Sequential(
        nn.Dropout(dropout_p),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model.to(DEVICE)

# ==========================================
# ⚖️ 5. ФУНКЦИИ ПОТЕРЬ
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets):
        n_classes = logits.size(1)
        log_probs = torch.log_softmax(logits, dim=1)
        targets_one_hot = torch.zeros_like(log_probs).scatter(1, targets.unsqueeze(1), 1)
        targets_smooth = targets_one_hot * (1 - self.smoothing) + (1 - targets_one_hot) * self.smoothing / (n_classes - 1)
        loss = -(targets_smooth * log_probs).sum(dim=1).mean()
        return loss

# ==========================================
# 🔄 6. CUTMIX
# ==========================================
def cutmix_data(x, y, alpha=1.0):
    r = np.random.beta(alpha, alpha)
    lam = max(r, 1 - r)
    idx = torch.randperm(x.size(0))
    x_b = x[idx]
    y_b = y[idx]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x_b[:, :, bbx1:bbx2, bby1:bby2]
    lam_adj = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))
    return x, y, y_b, lam_adj

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat); cut_h = int(H * cut_rat)
    cx = np.random.randint(W); cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W); bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W); bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

# ==========================================
# 🔄 7. ОБУЧЕНИЕ
# ==========================================
def run_experiment(cfg):
    print(f"\n🚀 {cfg['id']} | {cfg['mod']} | Loss: {cfg['loss']} | Mix: {cfg['mix']}")
    print(f"   База: LR={cfg['lr']:.0e}, WD={cfg['wd']:.0e}, Drop={cfg['drop']}, Freeze={cfg['freeze']}, Aug={cfg['aug']}, Resize={cfg['resize']}")

    gen = set_seed()

    mod_path = DATA_ROOT / cfg['mod']
    train_ds = datasets.ImageFolder(mod_path / "train", transform=get_transforms(cfg, True))
    val_ds   = datasets.ImageFolder(mod_path / "val",   transform=get_transforms(cfg, False))

    class_counts = torch.bincount(torch.tensor(train_ds.targets))
    sample_weights = [1.0 / class_counts[t] for t in train_ds.targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False, generator=gen)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    model = create_model(len(train_ds.classes), cfg['freeze'], cfg['drop'])
    optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['wd'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    if cfg['loss'] == 'focal':
        criterion = FocalLoss()
    elif cfg['loss'] == 'label_smooth':
        criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss()

    best_f1, best_epoch, patience_cnt = 0.0, 0, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch} Train", leave=False, colour='green')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            if cfg['mix'] == 'cutmix':
                inputs, y_a, y_b, lam = cutmix_data(inputs, labels)
                optimizer.zero_grad()
                # [ОШИБКА]: model(inputs) вызывается дважды в одной строке.
                # При включённом dropout каждый вызов даёт разные маски — градиент считается неверно.
                # Плюс итерация работает вдвое медленнее.
                # Исправить:
                #   outputs = model(inputs)
                #   loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
                loss = lam * criterion(model(inputs), y_a) + (1 - lam) * criterion(model(inputs), y_b)
            else:
                optimizer.zero_grad()
                loss = criterion(model(inputs), labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss = running_loss / len(train_ds)
        pbar.close()

        model.eval(); val_loss = 0.0; preds, labels_list = [], []
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Ep {epoch} Val", leave=False, colour='blue'):
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
                _, p = torch.max(outputs, 1)
                preds.extend(p.cpu().numpy()); labels_list.extend(labels.cpu().numpy())
        val_loss /= len(val_ds)
        f1 = f1_score(labels_list, preds, average='macro', zero_division=0)
        scheduler.step()

        if f1 > best_f1 + MIN_DELTA:
            best_f1, best_epoch, patience_cnt = f1, epoch, 0
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_best.pth")
            status = "⭐ BEST"
        else:
            patience_cnt += 1; status = f"wait {patience_cnt}/{PATIENCE}"

        print(f"✅ Ep {epoch:2d} | TrL: {train_loss:.4f} | VL: {val_loss:.4f} | F1: {f1:.4f} {status}")
        if patience_cnt >= PATIENCE:
            print(f"⏹️ Early Stop at {epoch}. Best F1: {best_f1:.4f}"); break

    torch.cuda.empty_cache()
    return {"id": cfg['id'], "modality": cfg['mod'], "loss": cfg['loss'], "mix": cfg['mix'],
            "best_f1": round(best_f1, 4), "best_epoch": best_epoch}

# ==========================================
# 🏁 8. ЗАПУСК
# ==========================================
if __name__ == "__main__":
    print(f"🖥️ Device: {DEVICE}")
    print("📌 СТРОГОЕ A/B ТЕСТИРОВАНИЕ: меняем ТОЛЬКО 1 параметр на модель")
    print("🔒 Seed, sampler, optimizer, scheduler, batch_size — ЗАФИКСИРОВАНЫ")

    results = []
    for cfg in EXPERIMENTS:
        results.append(run_experiment(cfg))

    df = pd.DataFrame(results)

    # Считаем дельты относительно базы
    for mod in ["ДС", "УФ"]:
        base_f1 = df[(df['id'].str.contains('Base')) & (df['modality']==mod)]['best_f1'].values[0]
        mask = df['modality'] == mod
        df.loc[mask, 'ΔF1'] = (df.loc[mask, 'best_f1'] - base_f1).round(4)

    df = df.sort_values(['modality', 'id'])
    print("\n" + "="*90)
    print("📊 ИТОГИ A/B ТЕСТИРОВАНИЯ")
    print("="*90)
    for mod in ["ДС", "УФ"]:
        print(f"\n🔹 {mod} (База ΔF1 = 0.0000)")
        mod_df = df[df['modality'] == mod]
        print(mod_df[['id', 'loss', 'mix', 'best_f1', 'ΔF1', 'best_epoch']].to_string(index=False))

    df.to_csv(SAVE_DIR / "ablation_strict.csv", index=False)
    print(f"\n💾 Таблица: {SAVE_DIR / 'ablation_strict.csv'}")

---
## Cell 3 — Class-Aware Augmentation

In [ ]:
"""
🔹 ЭКСПЕРИМЕНТ: Class-Aware Augmentation (Light vs Heavy)
🔹 Идея: Аугментировать только маленькие классы. Большие — минимально.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd
import numpy as np
import random
import time
from tqdm import tqdm
from pathlib import Path

# ==========================================
# 🔒 1. ФИКСАЦИЯ СИДА
# ==========================================

SEED = 42

def set_reproducible_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return torch.Generator().manual_seed(seed)

print(f"🔒 SEED={SEED} | Фиксация случайности активна")

# ==========================================
# 🔧 2. КОНФИГУРАЦИЯ
# ==========================================

DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./class_aug_test_light_vs_heavy")
SAVE_DIR.mkdir(exist_ok=True)

PATIENCE = 5
MIN_DELTA = 0.005
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_EPOCHS = 50

SMALL_CLASSES = [
    "Уголь,_уголь_с_прослоями_аргиллита",
    "Песчаник_с_прослоями_аргиллита",
    "Глинисто-карбонатная_порода",
    "Алевролит"
]

# ==========================================
# 📋 МАТРИЦА ЭКСПЕРИМЕНТОВ
# ==========================================
EXPERIMENTS = [
    # ДС
    {"id": "DS_StdAll",       "mod": "ДС", "class_aware": False}, # Все классы получают Light
    {"id": "DS_ClassAug",     "mod": "ДС", "class_aware": True},  # Малые -> Heavy, Большие -> Light

    # УФ
    {"id": "UF_StdAll",       "mod": "УФ", "class_aware": False}, # Все классы получают Light
    {"id": "UF_ClassAug",     "mod": "УФ", "class_aware": True},  # Малые -> Heavy, Большие -> Light
]

# ==========================================
# 🎨 3. ТРАНСФОРМЫ
# ==========================================
class PadToSquare:
    def __init__(self, fill=128): self.fill = fill
    def __call__(self, img):
        w, h = img.size
        if w == h: return img
        pad = abs(w - h) // 2
        return transforms.Pad((0, pad, 0, pad) if w > h else (pad, 0, pad, 0), fill=self.fill)(img)

def get_light_augmentations():
    """🟢 Минимальная аугментация для БОЛЬШИХ классов"""
    return [
        transforms.RandomHorizontalFlip(p=0.5)
    ]

def get_heavy_augmentations():
    """🔴 Усиленная аугментация для МАЛЫХ классов"""
    return [
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=10, translate=(0.15, 0.15)),
        transforms.RandomRotation(10)
    ]


def get_transforms(is_train, class_name, class_aware_flag):
    """
    Логика выбора аугментации:
    1. Если is_train=False -> только валидация (без аугментации).
    2. Если class_aware_flag=True И класс в SMALL_CLASSES -> HEAVY.
    3. В остальных случаях -> LIGHT.
    """
    ops = []

    if is_train:
        if class_aware_flag and class_name in SMALL_CLASSES:
            ops.extend(get_heavy_augmentations())
        else:
            ops.extend(get_light_augmentations())

    ops.append(PadToSquare(fill=128))
    ops.append(transforms.Resize((224, 224)))
    ops.append(transforms.ToTensor())
    ops.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    return transforms.Compose(ops)

# ==========================================
# 📦 4. КАСТОМНЫЙ DATASET
# ==========================================
class ClassAwareImageFolder(datasets.ImageFolder):
    """
    Переопределяем __getitem__, чтобы передать имя класса в функцию трансформации.
    """
    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = self.loader(path)
        class_name = self.classes[target]  # Получаем имя папки (класса)

        # Вызываем трансформ, передавая имя класса
        if self.transform is not None:
            sample = self.transform(sample, class_name, self.class_aware_flag)
        if self.target_transform is not None:
            target = self.target_transform(target)
        return sample, target

# ==========================================
# 🧠 5. МОДЕЛЬ
# ==========================================
def create_model(num_classes, freeze_mode, dropout_p):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_mode == "partial":
        for name, param in model.named_parameters():
            if any(x in name for x in ["conv1", "bn1", "layer1", "layer2"]):
                param.requires_grad = False
    elif freeze_mode == "full":
        for name, param in model.named_parameters():
            if "layer4" not in name and "fc" not in name:
                param.requires_grad = False

    model.fc = nn.Sequential(
        nn.Dropout(dropout_p),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model.to(DEVICE)

# ==========================================
# 🔄 6. ЦИКЛ ОБУЧЕНИЯ
# ==========================================
def run_experiment(cfg):
    print(f"\n🚀 {cfg['id']} | {cfg['mod']} | ClassAug: {cfg['class_aware']}")

    gen = set_reproducible_seed()
    mod_path = DATA_ROOT / cfg['mod']

    # 🔹 ОПРЕДЕЛЯЕМ ПАРАМЕТРЫ ПОД МОДАЛЬНОСТЬ
    lr = 5e-5 if cfg['mod'] == 'ДС' else 5e-4
    wd = 1e-3 if cfg['mod'] == 'ДС' else 1e-5
    dropout = 0.3
    freeze_mode = "none" if cfg['mod'] == 'ДС' else "partial"

    train_ds = ClassAwareImageFolder(mod_path / "train")
    # [ОШИБКА]: lambda возвращает объект Compose, а не применяет его к изображению.
    # get_transforms() возвращает transforms.Compose(...), поэтому sample станет
    # <transforms.Compose object> вместо torch.Tensor.
    # Модель упадёт с ошибкой типов при первом батче.
    # Исправить: lambda img, cls, flag: get_transforms(True, cls, flag)(img)
    #                                                                    ^^^^
    train_ds.transform = lambda img, cls, flag: get_transforms(True, cls, flag)
    # [ЗАМЕЧАНИЕ]: class_aware_flag добавляется как атрибут снаружи после создания объекта.
    # Если __getitem__ будет вызван до этой строки — AttributeError.
    # Лучше передавать флаг в __init__ ClassAwareImageFolder.
    train_ds.class_aware_flag = cfg['class_aware']

    val_ds = datasets.ImageFolder(mod_path / "val",
                                  transform=get_transforms(False, None, False))

    # 🔹 Sampler
    class_counts = torch.bincount(torch.tensor(train_ds.targets))
    sample_weights = [1.0 / class_counts[t] for t in train_ds.targets]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=gen
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False, generator=gen)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    model = create_model(len(train_ds.classes), freeze_mode, dropout)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_f1, best_epoch, patience_cnt = 0.0, 0, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch} Train", leave=False, colour='green')

        for inputs, labels in pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss = running_loss / len(train_ds)
        pbar.close()

        model.eval(); val_loss = 0.0; preds, labels_list = [], []
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Ep {epoch} Val", leave=False, colour='blue'):
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
                _, p = torch.max(outputs, 1)
                preds.extend(p.cpu().numpy()); labels_list.extend(labels.cpu().numpy())
        val_loss /= len(val_ds)
        f1 = f1_score(labels_list, preds, average='macro', zero_division=0)
        scheduler.step()

        if f1 > best_f1 + MIN_DELTA:
            best_f1, best_epoch, patience_cnt = f1, epoch, 0
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_best.pth")
            status = "⭐ BEST"
        else:
            patience_cnt += 1; status = f"wait {patience_cnt}/{PATIENCE}"

        print(f"✅ Ep {epoch:2d} | TrL: {train_loss:.4f} | VL: {val_loss:.4f} | F1: {f1:.4f} {status}")
        if patience_cnt >= PATIENCE:
            print(f"⏹️ Early Stop at {epoch}. Best F1: {best_f1:.4f}"); break

    torch.cuda.empty_cache()
    return {"id": cfg['id'], "modality": cfg['mod'], "class_aware": cfg['class_aware'],
            "best_f1": round(best_f1, 4), "best_epoch": best_epoch}

# ==========================================
# 🏁 ЗАПУСК
# ==========================================
if __name__ == "__main__":
    print(f"🖥️ Device: {DEVICE}")
    print("📌 ЭКСПЕРИМЕНТ: Light (Большие) vs Heavy (Малые)")
    print("🔹 Small classes list:", SMALL_CLASSES)

    results = []
    for cfg in EXPERIMENTS:
        results.append(run_experiment(cfg))

    df = pd.DataFrame(results)
    for mod in ["ДС", "УФ"]:
        base_f1 = df[(df['id'].str.contains('StdAll')) & (df['modality']==mod)]['best_f1'].values[0]
        mask = df['modality'] == mod
        df.loc[mask, 'ΔF1'] = (df.loc[mask, 'best_f1'] - base_f1).round(4)

    df = df.sort_values(['modality', 'id'])
    print("\n" + "="*90)
    print("📊 ИТОГИ: Light vs Heavy Augmentation")
    print("="*90)
    for mod in ["ДС", "УФ"]:
        print(f"\n🔹 {mod}")
        print(df[df['modality'] == mod][['id', 'class_aware', 'best_f1', 'ΔF1']].to_string(index=False))

    df.to_csv(SAVE_DIR / "class_aug_light_heavy.csv", index=False)